# **FRAMEWORK V7: NOTEBOOK DE EXTRACCIÓN DE DATOS CRUDOS**

CAPA - HIDROLOGICA

## **M0. Configuración General**

In [ ]:
import pandas as pd
import requests
from datetime import datetime

# ==============================================================
# CONFIGURACIÓN
# ==============================================================

NOMBRE_CAPA = "Información Hidrológica"
CODIGO_CAPA = "HID"
VERSION = "1.0"
RESPONSABLE = "Juan Carlos Riatiga"
FUENTE = "Datos Abiertos Colombia"
API_ID = "pt9a-aamx"

URL_API = f"https://www.datos.gov.co/resource/{API_ID}.json"

ARCHIVO_SALIDA = "01_Capa_Hidrologica_V1.xlsx"
ARCHIVO_AUDITORIA = "Auditoria_Capa_Hidrologica.xlsx" # Nueva constante para el archivo de auditoría

LIMITE = 50000

# Límites geográficos de la Cuenca del Río Bogotá

LAT_MIN = 4.0
LAT_MAX = 5.5

LON_MIN = -74.5
LON_MAX = -73.5

print("="*70)
print("FRAMEWORK V7")
print("CAPA 2 - INFORMACIÓN HIDROLÓGICA")
print("M0 - CONFIGURACIÓN GENERAL")
print("="*70)

print()

print("Fuente :", FUENTE)
print("API    :", API_ID)
print("Fecha  :", datetime.now())

print()

FRAMEWORK V7
CAPA 2 - INFORMACIÓN HIDROLÓGICA
M0 - CONFIGURACIÓN GENERAL

Fuente : Datos Abiertos Colombia
API    : pt9a-aamx
Fecha  : 2026-07-27 15:17:39.896218



## **M1. Definición de la Fuente Oficial**

La fuente oficial y la API son definidas en el módulo de Configuración General (M0) a través de las variables `FUENTE`, `API_ID` y `URL_API`.

## **M2. Extracción y Procesamiento de Datos**

In [ ]:
def extract_and_process_data(url_api, api_id, limite, lat_min, lat_max, lon_min, lon_max):
    """Extrae datos, valida columnas, convierte tipos y aplica filtro geográfico."""
    print("Conectando con la fuente oficial...")

    parametros = {
        "$limit": limite
    }

    response = requests.get(url_api,
                            params=parametros,
                            timeout=60)

    if response.status_code != 200:
        raise Exception(f"Error de conexión ({response.status_code})")

    df_niveles = pd.DataFrame(response.json())

    print("Conexión exitosa.")
    print()
    print("Registros descargados:", len(df_niveles))
    print("Variables encontradas:", len(df_niveles.columns))

    # ==========================================================
    # VALIDACIÓN DE COLUMNAS
    # ==========================================================

    columnas_requeridas = [
        "latitud",
        "longitud",
        "valorobservado",
        "fechaobservacion"
    ]

    for columna in columnas_requeridas:
        if columna not in df_niveles.columns:
            raise Exception(f"No existe la columna requerida: {columna}")

    # ==========================================================
    # CONVERSIÓN DE TIPOS
    # ==========================================================

    print()
    print("Convirtiendo tipos de datos...")

    df_niveles["latitud"] = pd.to_numeric(
        df_niveles["latitud"],
        errors="coerce"
    )

    df_niveles["longitud"] = pd.to_numeric(
        df_niveles["longitud"],
        errors="coerce"
    )

    df_niveles["valorobservado"] = pd.to_numeric(
        df_niveles["valorobservado"],
        errors="coerce"
    )

    df_niveles["fechaobservacion"] = pd.to_datetime(
        df_niveles["fechaobservacion"],
        errors="coerce"
    )

    # ==========================================================
    # FILTRADO GEOGRÁFICO
    # ==========================================================

    print()
    print("Aplicando filtro geográfico...")

    df_filtrado = df_niveles[
        (df_niveles["latitud"] >= lat_min) &
        (df_niveles["latitud"] <= lat_max) &
        (df_niveles["longitud"] >= lon_min) &
        (df_niveles["longitud"] <= lon_max)
    ].copy()

    df_filtrado = df_filtrado.sort_values(
        by="fechaobservacion"
    )

    return df_niveles, df_filtrado

## **M3. Reporte de Auditoría**

In [ ]:
def generate_audit_report(df_niveles, df_filtrado, archivo_auditoria):
    """Genera un reporte de auditoría y lo guarda en un archivo Excel."""
    print()
    print("="*70)
    print("DIAGNÓSTICO DE LA EXTRACCIÓN")
    print("="*70)

    audit_metrics = {
        "Registros originales": len(df_niveles),
        "Registros filtrados": len(df_filtrado)
    }

    if df_filtrado.empty:
        print("No existen registros para el área definida.")
        audit_metrics["Número de estaciones"] = 0
        audit_metrics["Número de municipios"] = 0
        audit_metrics["Número de departamentos"] = 0
        audit_metrics["Fecha inicial"] = "N/A"
        audit_metrics["Fecha final"] = "N/A"
        audit_metrics["Valores nulos"] = "N/A"
        audit_metrics["Duplicados"] = "N/A"
        # Add describe metrics as N/A or empty if no data
        for stat in ['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']:
            audit_metrics[f"Estadísticas de Nivel del Río - {stat}"] = "N/A"

    else:
        print()
        audit_metrics["Número de estaciones"] = df_filtrado["nombreestacion"].nunique()
        print("Número de estaciones :", audit_metrics["Número de estaciones"])

        audit_metrics["Número de municipios"] = df_filtrado["municipio"].nunique()
        print("Número de municipios :", audit_metrics["Número de municipios"])

        audit_metrics["Número de departamentos"] = df_filtrado["departamento"].nunique()
        print("Número de departamentos :", audit_metrics["Número de departamentos"])

        print()
        print("Cobertura temporal")
        print("-------------------")
        audit_metrics["Fecha inicial"] = df_filtrado["fechaobservacion"].min()
        print("Fecha inicial:", audit_metrics["Fecha inicial"])

        audit_metrics["Fecha final"] = df_filtrado["fechaobservacion"].max()
        print("Fecha final  :", audit_metrics["Fecha final"])

        print()
        print("Calidad del dato")
        print("----------------")
        audit_metrics["Valores nulos"] = df_filtrado.isnull().sum().sum()
        print("Valores nulos:", audit_metrics["Valores nulos"])

        audit_metrics["Duplicados"] = df_filtrado.duplicated().sum()
        print("Duplicados:", audit_metrics["Duplicados"])

        print()
        print("Estadísticas de Nivel del Río")
        print("-----------------------------")
        describe_series = df_filtrado["valorobservado"].describe()
        print(describe_series)
        for stat, value in describe_series.items():
            audit_metrics[f"Estadísticas de Nivel del Río - {stat}"] = value

    audit_df = pd.DataFrame(list(audit_metrics.items()), columns=['Métrica', 'Valor'])
    audit_df.to_excel(archivo_auditoria, index=False)
    print(f"\nReporte de auditoría generado en: {archivo_auditoria}")

## **M4. Exportación de Resultados**

In [ ]:
def export_results(df_filtrado, archivo_salida):
    """Exporta el DataFrame filtrado a un archivo Excel."""
    df_filtrado.to_excel(
        archivo_salida,
        index=False
    )
    print()
    print("="*70)
    print("ARCHIVO GENERADO")
    print("="*70)
    print(archivo_salida)

In [ ]:
# ==============================================================
# INICIO DEL PROCESO MODULAR
# ==============================================================

try:
    # M2. Extracción y Procesamiento de Datos
    df_niveles, df_filtrado = extract_and_process_data(
        URL_API, API_ID, LIMITE, LAT_MIN, LAT_MAX, LON_MIN, LON_MAX
    )

    # M3. Reporte de Auditoría
    generate_audit_report(df_niveles, df_filtrado, ARCHIVO_AUDITORIA)

    # M4. Exportación de Resultados
    export_results(df_filtrado, ARCHIVO_SALIDA)

    # M5. Resumen Final
    display_final_summary(df_filtrado)

except Exception as e:
    print()
    print("="*70)
    print("ERROR DURANTE LA EJECUCIÓN")
    print("="*70)
    print(str(e))

Conectando con la fuente oficial...
Conexión exitosa.

Registros descargados: 50000
Variables encontradas: 12

Convirtiendo tipos de datos...

Aplicando filtro geográfico...

DIAGNÓSTICO DE LA EXTRACCIÓN

Número de estaciones : 21
Número de municipios : 16
Número de departamentos : 2

Cobertura temporal
-------------------
Fecha inicial: 2006-08-24 16:00:00
Fecha final  : 2019-08-27 23:56:00

Calidad del dato
----------------
Valores nulos: 0
Duplicados: 0

Estadísticas de Nivel del Río
-----------------------------
count    5259.000000
mean        7.359947
std        29.723050
min         0.000000
25%         0.420000
50%         0.630000
75%         1.105000
max       299.000000
Name: valorobservado, dtype: float64

Reporte de auditoría generado en: Auditoria_Capa_Hidrologica.xlsx

ARCHIVO GENERADO
01_Capa_Hidrologica_V1.xlsx

Dimensiones finales:
(5259, 12)

Proceso finalizado correctamente.


## **M5. Resumen Final**

In [ ]:
def display_final_summary(df_filtrado):
    """Muestra las dimensiones finales y el mensaje de finalización."""
    print()
    print("Dimensiones finales:")
    print(df_filtrado.shape)
    print()
    print("Proceso finalizado correctamente.")

# **FRAMEWORK V7: NOTEBOOK DE EXTRACCIÓN DE DATOS CRUDOS**





In [ ]:
# ==============================================================
# FRAMEWORK V7
# CAPA 2 - INFORMACIÓN HIDROLÓGICA
# M1 - INGESTA DE DATOS
# Extracción y validación inicial de niveles del río
# Fuente: Datos Abiertos Colombia
# API: pt9a-aamx
# ==============================================================
# Autor: Juan Carlos Riatiga
# Versión: 1.0
# ==============================================================

import pandas as pd
import requests
from datetime import datetime

# ==============================================================
# CONFIGURACIÓN
# ==============================================================

NOMBRE_CAPA = "Información Hidrológica"

CODIGO_CAPA = "HID"

VERSION = "1.0"

RESPONSABLE = "Juan Carlos Riatiga"

FUENTE = "Datos Abiertos Colombia"

API_ID = "pt9a-aamx"

URL_API = f"https://www.datos.gov.co/resource/{API_ID}.json"

ARCHIVO_SALIDA = "01_Capa_Hidrologica_V1.xlsx"

LIMITE = 50000

# Límites geográficos de la Cuenca del Río Bogotá

LAT_MIN = 4.0
LAT_MAX = 5.5

LON_MIN = -74.5
LON_MAX = -73.5

# ==============================================================
# INICIO DEL PROCESO
# ==============================================================

print("="*70)
print("FRAMEWORK V7")
print("CAPA 2 - INFORMACIÓN HIDROLÓGICA")
print("M1 - INGESTA DE DATOS")
print("="*70)

print()

print("Fuente :", FUENTE)
print("API    :", API_ID)
print("Fecha  :", datetime.now())

print()

print("Conectando con la fuente oficial...")

try:

    # ==========================================================
    # DESCARGA
    # ==========================================================

    parametros = {

        "$limit": LIMITE

    }

    response = requests.get(URL_API,
                            params=parametros,
                            timeout=60)

    if response.status_code != 200:

        raise Exception(f"Error de conexión ({response.status_code})")

    df_niveles = pd.DataFrame(response.json())

    print("Conexión exitosa.")

    print()

    print("Registros descargados:", len(df_niveles))

    print("Variables encontradas:", len(df_niveles.columns))

    # ==========================================================
    # VALIDACIÓN DE COLUMNAS
    # ==========================================================

    columnas_requeridas = [

        "latitud",
        "longitud",
        "valorobservado",
        "fechaobservacion"

    ]

    for columna in columnas_requeridas:

        if columna not in df_niveles.columns:

            raise Exception(f"No existe la columna requerida: {columna}")

    # ==========================================================
    # CONVERSIÓN DE TIPOS
    # ==========================================================

    print()

    print("Convirtiendo tipos de datos...")

    df_niveles["latitud"] = pd.to_numeric(
        df_niveles["latitud"],
        errors="coerce"
    )

    df_niveles["longitud"] = pd.to_numeric(
        df_niveles["longitud"],
        errors="coerce"
    )

    df_niveles["valorobservado"] = pd.to_numeric(
        df_niveles["valorobservado"],
        errors="coerce"
    )

    df_niveles["fechaobservacion"] = pd.to_datetime(
        df_niveles["fechaobservacion"],
        errors="coerce"
    )

    # ==========================================================
    # FILTRADO GEOGRÁFICO
    # ==========================================================

    print()

    print("Aplicando filtro geográfico...")

    df_filtrado = df_niveles[

        (df_niveles["latitud"] >= LAT_MIN) &
        (df_niveles["latitud"] <= LAT_MAX) &
        (df_niveles["longitud"] >= LON_MIN) &
        (df_niveles["longitud"] <= LON_MAX)

    ].copy()

    df_filtrado = df_filtrado.sort_values(
        by="fechaobservacion"
    )

    # ==========================================================
    # AUDITORÍA INICIAL
    # ==============================================================

    print()

    print("="*70)
    print("DIAGNÓSTICO DE LA EXTRACCIÓN")
    print("="*70)

    print("Registros originales :", len(df_niveles))

    print("Registros filtrados  :", len(df_filtrado))

    if df_filtrado.empty:

        print()

        print("No existen registros para el área definida.")

    else:

        print()

        print("Número de estaciones :",
              df_filtrado["nombreestacion"].nunique())

        print("Número de municipios :",
              df_filtrado["municipio"].nunique())

        print("Número de departamentos :",
              df_filtrado["departamento"].nunique())

        print()

        print("Cobertura temporal")

        print("-------------------")

        print("Fecha inicial:",
              df_filtrado["fechaobservacion"].min())

        print("Fecha final  :",
              df_filtrado["fechaobservacion"].max())

        print()

        print("Calidad del dato")

        print("----------------")

        print("Valores nulos:",
              df_filtrado.isnull().sum().sum())

        print("Duplicados:",
              df_filtrado.duplicated().sum())

        print()

        print("Estadísticas de Nivel del Río")

        print("-----------------------------")

        print(df_filtrado["valorobservado"].describe())

        # ======================================================
        # EXPORTACIÓN
        # ======================================================

        df_filtrado.to_excel(
            ARCHIVO_SALIDA,
            index=False
        )

        print()

        print("="*70)

        print("ARCHIVO GENERADO")

        print("="*70)

        print(ARCHIVO_SALIDA)

        print()

        print("Dimensiones finales:")

        print(df_filtrado.shape)

        print()

        print("Proceso finalizado correctamente.")

except Exception as e:

    print()

    print("="*70)

    print("ERROR DURANTE LA EJECUCIÓN")

    print("="*70)

    print(str(e))

FRAMEWORK V7
CAPA 2 - INFORMACIÓN HIDROLÓGICA
M1 - INGESTA DE DATOS

Fuente : Datos Abiertos Colombia
API    : pt9a-aamx
Fecha  : 2026-07-14 00:44:38.426850

Conectando con la fuente oficial...
Conexión exitosa.

Registros descargados: 50000
Variables encontradas: 12

Convirtiendo tipos de datos...

Aplicando filtro geográfico...

DIAGNÓSTICO DE LA EXTRACCIÓN
Registros originales : 50000
Registros filtrados  : 5259

Número de estaciones : 21
Número de municipios : 16
Número de departamentos : 2

Cobertura temporal
-------------------
Fecha inicial: 2006-08-24 16:00:00
Fecha final  : 2019-08-27 23:56:00

Calidad del dato
----------------
Valores nulos: 0
Duplicados: 0

Estadísticas de Nivel del Río
-----------------------------
count    5259.000000
mean        7.359947
std        29.723050
min         0.000000
25%         0.420000
50%         0.630000
75%         1.105000
max       299.000000
Name: valorobservado, dtype: float64

ARCHIVO GENERADO
01_Capa_Hidrologica_V1.xlsx

Dimensiones